# 03 — RAG Optimization

Improve retrieval and generation quality with hybrid search, re-ranking, prompt engineering, and evaluation.

## What you'll learn
- Hybrid search (keyword + semantic)
- Re-ranking retrieved results
- Prompt engineering for grounded answers
- Evaluating RAG quality
- Common failure modes and fixes

In [ ]:
# !pip install langchain langchain-openai langchain-pinecone langchain-community pinecone-client python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
print("Ready.")

## 1. Setup — Create Knowledge Base

We build a small knowledge base to test optimization techniques.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain_openai import ChatOpenAIEmbeddings, ChatChatOpenAI
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone
import numpy as np

# Sample knowledge base documents
kb_docs = [
    Document(page_content="RAG reduces hallucination by grounding LLM responses in retrieved source documents. The model generates answers based on context provided from the knowledge base rather than its training data alone.", metadata={"source": "rag-overview", "topic": "rag"}),
    Document(page_content="Pinecone is a managed vector database optimized for similarity search. It supports metadata filtering, namespaces, and hybrid search. Pinecone handles indexing and scaling automatically.", metadata={"source": "pinecone-guide", "topic": "vectordb"}),
    Document(page_content="Chunk size significantly impacts retrieval quality. Chunks that are too small lose context, while chunks that are too large reduce precision. Recommended range is 200-1000 characters with 10-20% overlap.", metadata={"source": "chunking-best-practices", "topic": "chunking"}),
    Document(page_content="Hybrid search combines keyword-based retrieval (BM25) with semantic search. Keyword search excels at exact matches, names, and codes. Semantic search captures meaning and intent. Combining both improves recall.", metadata={"source": "hybrid-search-docs", "topic": "retrieval"}),
    Document(page_content="Re-ranking improves retrieval precision by scoring initial results with a cross-encoder model. The re-ranker considers query-document pairs together, producing more accurate relevance scores than embedding similarity alone.", metadata={"source": "reranking-guide", "topic": "retrieval"}),
    Document(page_content="GPT-4o Mini is Google's fast multimodal model. It supports text and image inputs. For RAG, Gemini provides strong reasoning with low latency. Use temperature=0 for deterministic, fact-based answers.", metadata={"source": "gemini-docs", "topic": "llm"}),
    Document(page_content="Embedding models convert text to vectors. text-embedding-3-small produces 768-dimensional vectors. Quality varies by domain — general models may underperform on specialized text like legal or medical documents.", metadata={"source": "embeddings-guide", "topic": "embeddings"}),
    Document(page_content="The lost-in-the-middle problem occurs when LLMs fail to use information placed in the middle of long contexts. Solutions include re-ranking to place most relevant chunks at the start/end, or using shorter context windows.", metadata={"source": "llm-quirks", "topic": "llm"}),
]

# Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(kb_docs)

# Initialize components
embedding_model = ChatOpenAIEmbeddings(model="models/text-embedding-3-small")
llm = ChatChatOpenAI(model="gpt-4o-mini", temperature=0)

# Connect to Pinecone
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
INDEX_NAME = "rag-optimization"

existing = [idx.name for idx in pc.list_indexes()]
if INDEX_NAME not in existing:
    pc.create_index(
        name=INDEX_NAME, dimension=768, metric="cosine",
        spec={"serverless": {"cloud": "aws", "region": "us-east-1"}}
    )

# Upsert
vector_store = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embedding_model,
    index_name=INDEX_NAME,
    namespace="optimized"
)

print(f"Knowledge base: {len(chunks)} chunks indexed.")

## 2. Baseline Retrieval

Start with standard similarity search as a baseline.

In [ ]:
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA

retriever = vector_store.as_retriever(
    search_kwargs={"k": 3, "namespace": "optimized"}
)

# Basic prompt
basic_prompt = PromptTemplate(
    template="""Answer the question based on the context.

Context:
{context}

Question: {question}

Answer:""",
    input_variables=["context", "question"]
)

baseline_chain = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=retriever,
    return_source_documents=True, chain_type_kwargs={"prompt": basic_prompt}
)

# Test query
test_query = "How does Pinecone handle similarity search?"
result = baseline_chain.invoke({"query": test_query})

print(f"Q: {test_query}")
print(f"A: {result['result']}")
print("\nSources:")
for doc in result["source_documents"]:
    print(f"  - {doc.metadata['source']}: {doc.page_content[:100]}...")

## 3. Hybrid Search

Combine BM25 keyword search with semantic search. BM25 catches exact term matches; semantic catches meaning.

In [ ]:
from langchain.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

# BM25 keyword retriever on the chunks
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 3

# Semantic retriever
semantic_retriever = vector_store.as_retriever(
    search_kwargs={"k": 3, "namespace": "optimized"}
)

# Ensemble: 30% BM25 + 70% semantic
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, semantic_retriever],
    weights=[0.3, 0.7]
)

hybrid_chain = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=hybrid_retriever,
    return_source_documents=True, chain_type_kwargs={"prompt": basic_prompt}
)

# Compare results
queries = [
    "How does Pinecone handle similarity search?",
    "What causes the lost-in-the-middle problem?",
    "How should I choose chunk size?"
]

for q in queries:
    baseline = baseline_chain.invoke({"query": q})
    hybrid = hybrid_chain.invoke({"query": q})
    
    print(f"Q: {q}")
    print(f"  Baseline: {baseline['result'][:150]}...")
    print(f"  Hybrid:   {hybrid['result'][:150]}...")
    print()

## 4. Re-ranking

After initial retrieval, re-rank results using a cross-encoder for higher precision. We'll use a simulated reranking approach with embedding similarity as a stand-in.

For production, use Cohere Rerank, a cross-encoder, or ColBERT.

In [ ]:
def rerank_with_query(query, docs, embeddings_model, top_n=3):
    """Simulate re-ranking by computing query-document similarity directly
    (in production, use a cross-encoder model for true re-ranking)."""
    query_emb = embeddings_model.embed_query(query)
    doc_embs = embeddings_model.embed_documents([d.page_content for d in docs])
    
    similarities = []
    for doc, emb in zip(docs, doc_embs):
        sim = np.dot(query_emb, emb) / (np.linalg.norm(query_emb) * np.linalg.norm(emb))
        similarities.append((doc, sim))
    
    similarities.sort(key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in similarities[:top_n]]


def reranked_retrieval(query, base_retriever, embedding_model, top_k=5, top_n=3):
    """Retrieve more results, then re-rank down to top_n."""
    initial_docs = base_retriever.invoke(query)
    reranked = rerank_with_query(query, initial_docs, embedding_model, top_n=top_n)
    return reranked


# Compare: baseline vs reranked
query = "How does Pinecone handle similarity search?"

baseline_docs = retriever.invoke(query)
reranked_docs = reranked_retrieval(query, retriever, embedding_model, top_k=5, top_n=3)

print(f"Q: {query}\n")
print("Baseline retrieval sources:")
for doc in baseline_docs:
    print(f"  - {doc.metadata['source']}")

print("\nReranked sources:")
for doc in reranked_docs:
    print(f"  - {doc.metadata['source']}")

## 5. Prompt Engineering for RAG

The prompt template controls how the LLM uses retrieved context. Compare a basic prompt vs an optimized one.

In [ ]:
# Optimized RAG prompt
optimized_prompt = PromptTemplate(
    template="""You are a helpful assistant. Answer the question using ONLY the provided context.
If the context does not contain enough information to answer, say "I don't have enough information to answer this."
Do not make up or infer information that isn't in the context.
When possible, reference which part of the context supports your answer.

Context:
{context}

Question: {question}

Provide a clear, concise answer:""",
    input_variables=["context", "question"]
)

# Build chain with optimized prompt
optimized_chain = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=retriever,
    return_source_documents=True, chain_type_kwargs={"prompt": optimized_prompt}
)

# Compare
test_q = "What is the recommended chunk size for RAG?"

basic_result = baseline_chain.invoke({"query": test_q})
optimized_result = optimized_chain.invoke({"query": test_q})

print(f"Q: {test_q}\n")
print("Basic prompt:")
print(f"  {basic_result['result']}\n")
print("Optimized prompt:")
print(f"  {optimized_result['result']}")

## 6. Evaluating RAG Quality

Measure retrieval quality and answer quality using simple metrics.

| Metric | What It Measures |
|---|---|
| **Context Precision** | Are retrieved chunks relevant? |
| **Answer Relevance** | Does the answer address the question? |
| **Faithfulness** | Is the answer grounded in context? |
| **Context Recall** | Did retrieval capture all relevant info? |

In [ ]:
def evaluate_retrieval(retriever, queries, expected_keywords):
    """Evaluate retrieval quality via keyword recall."""
    scores = []
    for query, keywords in zip(queries, expected_keywords):
        docs = retriever.invoke(query)
        retrieved_text = " ".join(d.page_content for d in docs)
        found = sum(1 for kw in keywords if kw.lower() in retrieved_text.lower())
        scores.append(found / len(keywords))
    return np.mean(scores)


eval_queries = [
    "How does RAG reduce hallucination?",
    "What is the lost-in-the-middle problem?",
    "How does hybrid search work?"
]

eval_keywords = [
    ["hallucination", "retrieved", "grounding"],
    ["middle", "context", "re-ranking"],
    ["bm25", "keyword", "semantic"]
]

baseline_score = evaluate_retrieval(retriever, eval_queries, eval_keywords)
hybrid_score = evaluate_retrieval(hybrid_retriever, eval_queries, eval_keywords)

print(f"Retrieval Quality (keyword recall):")
print(f"  Baseline (semantic):  {baseline_score:.1%}")
print(f"  Hybrid (BM25+sem):    {hybrid_score:.1%}")

In [ ]:
def evaluate_faithfulness(chain, queries, context_keywords):
    """Check if answers mention keywords from the retrieved context."""
    scores = []
    for query, keywords in zip(queries, context_keywords):
        result = chain.invoke({"query": query})
        answer = result["result"].lower()
        found = sum(1 for kw in keywords if kw.lower() in answer)
        scores.append(found / len(keywords))
    return np.mean(scores)

answer_keywords = [
    ["source", "documents", "context"],
    ["middle", "position"],
    ["keyword", "semantic", "combine"]
]

baseline_faithfulness = evaluate_faithfulness(baseline_chain, eval_queries, answer_keywords)
optimized_faithfulness = evaluate_faithfulness(optimized_chain, eval_queries, answer_keywords)

print(f"Answer Faithfulness:")
print(f"  Basic prompt:    {baseline_faithfulness:.1%}")
print(f"  Optimized prompt: {optimized_faithfulness:.1%}")

## 7. Common RAG Failure Modes

### Failure 1: No Relevant Context Retrieved
**Symptom:** Model says "I don't know" or gives generic answers.
**Fix:** Increase `k`, try hybrid search, expand the knowledge base.

In [ ]:
# Demonstrate: query that may not match well
edge_query = "vector database scaling"

docs_k3 = vector_store.similarity_search(edge_query, k=3, namespace="optimized")
docs_k6 = vector_store.similarity_search(edge_query, k=6, namespace="optimized")

print(f"Query: '{edge_query}'")
print(f"k=3: {len(docs_k3)} results, sources: {[d.metadata['source'] for d in docs_k3]}")
print(f"k=6: {len(docs_k6)} results, sources: {[d.metadata['source'] for d in docs_k6]}")

### Failure 2: Retrieved Context Is Not Relevant
**Symptom:** Answer is wrong despite retrieval succeeding.
**Fix:** Add metadata filtering, improve chunking, use re-ranking.

In [ ]:
# Metadata filtering: only retrieve from a specific topic
filtered_docs = vector_store.similarity_search(
    "How does RAG work?",
    k=3,
    filter={"topic": "rag"},
    namespace="optimized"
)

print("Filtered to topic='rag':")
for doc in filtered_docs:
    print(f"  - {doc.metadata['source']} (topic: {doc.metadata['topic']})")

### Failure 3: Lost in the Middle
**Symptom:** Model ignores relevant context placed in the middle of long prompts.
**Fix:** Place most relevant chunks first/last, use shorter contexts, re-rank.

In [ ]:
# Simulate: demonstrate context ordering effect
def build_context_with_ordering(docs, target_source):
    """Place target doc at start vs middle to show ordering effect."""
    target = [d for d in docs if d.metadata['source'] == target_source][0]
    others = [d for d in docs if d.metadata['source'] != target_source]
    
    # Target at start
    start_context = target.page_content + "\n\n" + "\n\n".join(d.page_content for d in others)
    # Target in middle
    mid = len(others) // 2
    mid_docs = others[:mid] + [target] + others[mid:]
    mid_context = "\n\n".join(d.page_content for d in mid_docs)
    
    return start_context, mid_context

docs = vector_store.similarity_search("What is Pinecone?", k=5, namespace="optimized")
start_ctx, mid_ctx = build_context_with_ordering(docs, "pinecone-guide")

print(f"Context with Pinecone at START: {len(start_ctx)} chars")
print(f"Context with Pinecone in MIDDLE: {len(mid_ctx)} chars")
print("\nRecommendation: Always place most relevant context at the beginning.")

## 8. Putting It All Together

Build a production-ready RAG chain with all optimizations.

In [ ]:
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser

# Production prompt
production_prompt = PromptTemplate(
    template="""Answer the question using ONLY the provided context.
If the context doesn't contain enough information, say so clearly.
Do not infer or make up information.

Context:
{context}

Question: {question}

Answer:""",
    input_variables=["context", "question"]
)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

# Production RAG chain
production_chain = (
    {"context": hybrid_retriever | format_docs, "question": RunnablePassthrough()}
    | production_prompt
    | llm
    | StrOutputParser()
)

# Test
response = production_chain.invoke("How does RAG reduce hallucination?")
print("Production RAG Answer:")
print(response)

## Summary

| Optimization | Impact | Complexity |
|---|---|---|
| Hybrid search | Better recall for exact terms | Low |
| Re-ranking | Higher precision | Medium |
| Optimized prompts | More faithful answers | Low |
| Metadata filtering | More targeted retrieval | Low |
| Context ordering | Avoids lost-in-the-middle | Low |

**Key takeaway:** Start simple, measure quality, then optimize. Most improvements come from better chunking and prompt engineering, not complex infrastructure.

Next: exercises in `exercises/exercise-01.md`.